In [ ]:
# Подключаем библиотеку pandas (она умеет работать с таблицами Excel)
import pandas as pd

# --- НАСТРОЙКИ ---
# Пишем название файла, который будем открывать
file_path = 'reviews.xlsx' 

# Список кусочков хороших слов. Если они есть в тексте — это плюс балл
positive_words = ['хорош', 'супер', 'класс', 'отличн', 'понравил']

# Список кусочков плохих слов. Если они есть — минус балл
negative_words = ['плох', 'ужас', 'слом', 'брак', 'грустно', 'не работ', 'разочар']

# Слова-отрицания. Помогают понять, когда плохое слово используется "наоборот"
negations = ['не ', 'нет ']

# Создаем специальную коробочку (функцию), куда положим текст и получим ответ: хороший он или плохой
def analyze_sentiment(text): 
    # Если ячейка пустая или там только пробелы, сразу говорим, что отзыв нейтральный
    if text is None or str(text).strip() == "":  
        return 'Нейтрально'
        
    # Берем наш текст и делаем все буквы маленькими, чтобы не путать "Супер" и "супер"
    text_lower = str(text).lower() 
    
    # Заводим счетчик баллов. В начале у нас 0 очков
    score = 0

    # ИДЕМ ИСКАТЬ ХОРОШИЕ СЛОВА:
    # Проходим по списку каждого хорошего слова
    for word in positive_words:
        # Если кусочек этого слова нашелся внутри отзыва
        if word in text_lower:
            # Добавляем один балл к нашему счету
            score += 1
                                                
    # ИДЕМ ИСКАТЬ ПЛОХИЕ СЛОВА:
    # Проходим по списку каждого плохого слова
    for word in negative_words:
         # Если кусочек плохого слова нашелся в тексте
         if word in text_lower:
             # Сначала думаем, что отрицания перед ним нет
             is_negated = False
             
             # Ищем, на каком месте в тексте стоит это плохое слово
             neg_index = text_lower.find(word)
             
             # Если слово вообще нашли (индекс не равен -1)
             if neg_index != -1:
                 # Берем несколько букв ДО этого плохого слова, чтобы проверить частицу "не"
                 prefix = text_lower[max(0, neg_index-4):neg_index] 
                 
                 # Проверяем каждое наше слово-отрицание
                 for neg in negations:
                      # Если частица "не" оказалась прямо перед плохим словом
                      if neg in prefix:
                          # Ставим галочку: да, тут отрицание!
                          is_negated = True
                          break # Выходим из проверки частиц, всё понятно
                          
             # Если мы НЕ нашли отрицание перед плохим словом...
             if not is_negated:
                 # ...тогда вычитаем балл из нашего счета
                 score -= 1

    # РЕШАЕМ, КАКОЙ ЭТО ОТЗЫВ:
    # Если баллов больше нуля — значит, было много хорошего
    if score > 0:
        return 'Позитив'
    # Если меньше нуля — перевесили плохие слова
    elif score < 0:
        return 'Негатив'
    # Если ровно ноль — хорошее уравновесило плохое
    else:
        return 'Нейтрально'
   
# ОСНОВНАЯ ЧАСТЬ ПРОГРАММЫ:
try:
   # Открываем нашу таблицу Excel как табличку в памяти компьютера
   df = pd.read_excel(file_path)
   
   # Ищет колонку, которая похожа на отзывы (ищет слова review или отзыв)
   review_column_candidates = [col for col in df.columns if 'review' in col.lower() or 'отзыв' in col.lower()]
   
   # Если такой колонки вообще не нашлось
   if not review_column_candidates:
       # Печатаем ошибку для пользователя
       print("Ошибка: Не удалось найти колонку с отзывами.")
       # Сразу выключаем программу, дальше идти нет смысла
       exit()
       
   # Запоминаем точное имя той колонки, которую нашли самой первой
   review_col_name = review_column_candidates[0]
   
   # Применяем нашу функцию ко всем строчкам найденной колонки
   # Результат записываем в новую колонку справа под названием 'sentiment'
   df['sentiment'] = df[review_col_name].apply(analyze_sentiment)
                                   
   # Говорим пользователю, что работа сделана
   print("Анализ завершен!")
   # Показываем первые 5 строк таблицы, чтобы можно было глазами проверить результат
   print(df.head()) 
   
   # --- РИСУЕМ ГРАФИК И СОХРАНЯЕМ ЕГО ---
   # Подключаем инструмент для рисования красивых графиков
   import plotly.express as px
   
   # Просим нарисовать столбчатую диаграмму (bar chart)
   fig = px.bar(
       # Из какой таблицы брать данные
       df, 
       # Что рисовать по горизонтали (названия категорий)
       x='sentiment', 
       # Каким цветом красить столбики
       color='sentiment',
       # Какой заголовок написать сверху картинки
       title='Распределение тональности отзывов о продукте',
       # Чтобы прямо на столбиках были написаны цифры количества
       text_auto=True,
       # Задаем строгий порядок столбцов слева направо
       category_orders={'sentiment': ['Позитив', 'Нейтрально', 'Негатив']} 
   )
   
   # Убираем лишнюю рамочку со списком цветов сбоку
   fig.update_layout(showlegend=False)
   
   # Придумываем имя файлу, куда сохраним картинку
   output_filename = 'tonalnost_otzyvov.html'
   
   # Сохраняем график не просто как картинку, а как интерактивную веб-страницу
   fig.write_html(output_filename)
   
   # Сообщаем человеку, что файл готов
   print(f"\nГрафик успешно сохранен в файл '{output_filename}'")
   print("Откройте этот файл двойным кликом в любом браузере (Chrome, Firefox, Safari), чтобы увидеть диаграмму.")
   
# ЕСЛИ ФАЙЛА НЕТ НА ДИСКЕ:
except FileNotFoundError:
    # Пишем понятное сообщение, почему ничего не работает
    print(f"Ошибка: Файл '{file_path}' не найден.")
    print("Убедись, что файл лежит в той же папке, где запущен скрипт.")
# ЕСЛИ СЛУЧИЛАСЬ ЛЮБАЯ ДРУГАЯ ОШИБКА:
except Exception as e:
    # Показываем технический текст ошибки, чтобы программист понял, что сломалось
    print(f"Произошла непредвиденная ошибка: {e}")